# Incoorporating new files

## Setup

Add your data path here (`base_path`)

In [40]:
# Add path
import sys
sys.path.append('../')
from util import utilities as util
from util.data_loader import DataLoader
import os
import shutil
import pandas as pd
from tqdm import tqdm

# Path to the new data
base_path = r"E:\OPFA"

## Utilities

In [38]:
def load_new_metadata():
    metadata = util.load_metadata(animal_info_path=base_path+r"\animal_info.csv",
                    trial_info_path=base_path+r"\trial_info.csv",)
    return metadata

def get_date_format(date):
    ''' Convert YYYYMMDD to "YYYY-MM-DD" '''
    date_str = str(date)
    date_formatted = f"{date_str[0:4]}-{date_str[4:6]}-{date_str[6:8]}"
    return date_formatted

def generate_source_file_path(dl, data_type = 'behVideo'):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    date_formatted = get_date_format(dl.date)
    date_folder_path = os.path.join(base_path, 'data', date_formatted)

    if data_type == 'triggerLoc':
        return os.path.join(date_folder_path, 'locations.csv')
    elif data_type == 'behVideo':
        return os.path.join(date_folder_path, 'movie', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}.avi")
    elif data_type == 'trackInfo':
        return os.path.join(date_folder_path, 'movie', 'tracking', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}_positions.csv")
    elif data_type == 'exampleImg':
        return os.path.join(date_folder_path, 'movie', 'tracking', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}_trace.png")
    else:
        print('Unknown data type')

def generate_desti_file_path(dl, data_type = 'behVideo'):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    if data_type == 'triggerLoc':
        return dl.generate_filepath('triggerLoc', 'rawdata', 'behav', 'csv')
    if data_type == 'trackInfo':
        return dl.generate_filepath('position', 'derivatives', 'behav', 'csv')
    if data_type == 'exampleImg':
        return dl.generate_filepath('trace', 'derivatives', 'behav', 'png')
    if data_type == 'behVideo':
        return dl.generate_filepath('video', 'rawdata', 'behav', 'avi')
    else: 
        print('Unknown data type')

def check_source_file_exists(dl, data_type):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    file = generate_source_file_path(dl, data_type)
    if not os.path.isfile(file):
        print(f'{dl.id}-{dl.date}-trial_{dl.trial_id}: {data_type} missing')
        return False
    return True


In [11]:
metadata = load_new_metadata()

## Check existance of all files  

Files for each trial:  
    * trackInfo: `*.positions.csv`  
    * exampleImg: `*.trace.png`  
    * behVideo: `*.avi`  
Files for each day:  
    * triggerLoc: `*.locations.csv`  

In [37]:
data_types = {'trackInfo', 'exampleImg', 'behVideo', 'triggerLoc'}

flag = True
for rowIdx in range(len(metadata)):
    dl = DataLoader(metadata_ses=metadata.iloc[rowIdx])
    for data_type in data_types:
        if not check_source_file_exists(dl, data_type):
            flag = False

if flag:
    print('All source files exist!')

BD-20251125-trial_4: trackInfo missing
BD-20251125-trial_4: exampleImg missing
BD-20251125-trial_4: behVideo missing


## Move data

In [41]:
for rowIdx in tqdm(range(len(metadata))):
    dl = DataLoader(metadata_ses=metadata.iloc[rowIdx])
    for data_type in data_types:
        source_file_path = generate_source_file_path(dl, data_type)
        desti_file_path = generate_desti_file_path(dl, data_type)

        os.makedirs(os.path.dirname(desti_file_path), exist_ok=True)

        if not os.path.isfile(source_file_path):
            print(f'{dl.id}-{dl.date}-trial_{dl.trial_id}: {data_type} missing')
            continue

        if data_type == 'triggerLoc':
            loc = pd.read_csv(source_file_path)
            loc = loc.rename(columns={
                'Animal': 'id',
                'Well_row': 'well_row',
                'Well_col': 'well_col',
                'Valence': 'valence',
            })
            loc = loc[loc['id'] == dl.id].reset_index(drop=True)
            loc.to_csv(desti_file_path, index=False)
        else:
            shutil.copy2(source_file_path, desti_file_path)
        

 35%|███▍      | 192/549 [00:58<01:47,  3.31it/s]

BD-20251125-trial_4: trackInfo missing
BD-20251125-trial_4: exampleImg missing
BD-20251125-trial_4: behVideo missing


100%|██████████| 549/549 [02:24<00:00,  3.79it/s]
